In [ ]:
from pathlib import Path
import sys


def find_postprocessing_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "powerflow" / "comparison_data.py").exists():
            return candidate
        nested = candidate / "GridExpand" / "5.postprocessing"
        if (nested / "powerflow" / "comparison_data.py").exists():
            return nested
    raise FileNotFoundError(
        "Could not find GridExpand/5.postprocessing from the notebook working directory."
    )


POSTPROCESSING_DIR = find_postprocessing_dir()
if str(POSTPROCESSING_DIR) not in sys.path:
    sys.path.insert(0, str(POSTPROCESSING_DIR))

from expansion.notebook_workflow import (
    load_powerflow_cutoff_comparison,
    prepare_expansion_analysis,
)
from powerflow.comparison_data import powerflow_distribution_similarity_summary
from plotting.powerflow_asset_plots import (
    plot_cable_max_loading_ecdf,
    plot_powerflow_asset_cutoff_overview,
    plot_powerflow_asset_cutoff_overview_static,
)
from plotting.powerflow_io import save_plotly_figure


In [ ]:
AGS = "9474126"
PLZ = 91301
SCENARIO_PREFIX = "forchheim_paired_battery_tsam"
EXPECTED_GRID_COUNTS = {"Synthetic": 83, "Real SWF": 88}
EXCLUDED_REAL_LV_IDS = (113,)
COLORS = {
    "Synthetic": "#335C81",
    "Real SWF": "#D95D39",
}
PLOTTING_DIR = POSTPROCESSING_DIR / "output" / "notebook_plots" / SCENARIO_PREFIX


In [ ]:
analysis_context = prepare_expansion_analysis(
    scenario_prefix=SCENARIO_PREFIX,
    ags=AGS,
    expected_grid_counts=EXPECTED_GRID_COUNTS,
    real_plz=PLZ,
)
pre_label = analysis_context["stage_labels"]["pre"]
synthetic_pre_spec = {pre_label: analysis_context["synthetic_specs"][pre_label]}
real_pre_spec = {pre_label: analysis_context["real_specs"][pre_label]}
comparison_data = load_powerflow_cutoff_comparison(
    synthetic_specs=synthetic_pre_spec,
    real_specs=real_pre_spec,
    stage_order=[pre_label],
    ags=AGS,
    real_plz=PLZ,
    excluded_real_lv_ids=EXCLUDED_REAL_LV_IDS,
)
percentile_profile = comparison_data["profile"]
coverage = comparison_data["coverage_summary"]
asset_summary = comparison_data["asset_summary"]
excluded_real_grids = comparison_data["excluded_real_grids"]


In [ ]:
display(coverage)
display(asset_summary)
if not excluded_real_grids.empty:
    print("Explicitly excluded real SWF grids:")
    display(excluded_real_grids)


In [ ]:
similarity_summary = powerflow_distribution_similarity_summary(percentile_profile)
similarity_summary

### Line Maximum Loading ECDF


In [ ]:
cable_max_loading_ecdf_fig = plot_cable_max_loading_ecdf(
    percentile_profile,
    group_col="comparison_group",
    color_map=COLORS,
    title="Paired Status-Quo Cable Maximum Loading ECDF",
    show=False,
)
display(cable_max_loading_ecdf_fig)


In [ ]:
ASSET_CUTOFF_FILTER_SCOPE = "grid"  # "asset", "grid"
asset_cutoff_overview_fig = plot_powerflow_asset_cutoff_overview(
    percentile_profile,
    group_col="comparison_group",
    color_map=COLORS,
    asset_cutoff_percentiles=(1.0, 0.99, 0.95, 0.90),
    y_axis_limits=(80, 100, 0.85),
    center_stat="mean",  # "median" or "mean"
    show_band=False,
    worst_asset_per_grid=True,
    filter_scope=ASSET_CUTOFF_FILTER_SCOPE,  # "asset", "grid"
    title=f"Paired Status-Quo: Synthetic vs Real SWF - Retained {ASSET_CUTOFF_FILTER_SCOPE.capitalize()} Cutoff",
)


In [ ]:
OUTPUT_DIR = POSTPROCESSING_DIR / "output" / "plots" / "asset_powerflow"
saved_asset_percentile_paths = save_plotly_figure(
    asset_cutoff_overview_fig,
    OUTPUT_DIR / "asset-percentiles",
    formats=("png", "svg"),
    width=1500,
    height=860,
    scale=2.0,
    active_slider_step=f"Show {ASSET_CUTOFF_FILTER_SCOPE} cutoffs through P95",
)
saved_asset_percentile_paths


In [ ]:
OUTPUT_DIR = POSTPROCESSING_DIR / "output" / "plots" / "asset_powerflow"
static_asset_cutoff_overview_fig = plot_powerflow_asset_cutoff_overview_static(
    percentile_profile,
    group_col="comparison_group",
    color_map=COLORS,
    asset_cutoff_percentile=0.95,
    y_axis_limits=(100, 100, 0.85),
    center_stat="mean",
    show_band=False,
    worst_asset_per_grid=True,
    filter_scope=ASSET_CUTOFF_FILTER_SCOPE,
    title=f"Paired Status-Quo: Synthetic vs Real SWF - Retained {ASSET_CUTOFF_FILTER_SCOPE.capitalize()} Cutoff",
    save_path=OUTPUT_DIR / "asset-percentiles-static",
    save_formats=("svg", "pdf"),
)
